<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione4/Lezione4_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 4 — RAG: Conoscenza Personalizzata

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Giovedì 28/05/2026

---

### 🎯 Obiettivi
- ✅ Capire la pipeline RAG completa
- ✅ Indicizzare un PDF con ChromaDB
- ✅ Implementare la ricerca semantica
- ✅ Integrare RAG nel chatbot esistente

In [1]:
# Setup
!pip install anthropic chromadb pypdf sentence-transformers -q
from google.colab import userdata
import anthropic, os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

def chiedi_claude(domanda, system=None, max_tokens=800):
    params = {"model":"claude-haiku-4-5-20251001","max_tokens":max_tokens,
              "messages":[{"role":"user","content":domanda}]}
    if system: params["system"] = system
    return client.messages.create(**params).content[0].text

print("✅ Setup completato!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

---
## 1. Crea un documento di test

Per l'esercizio creiamo un documento di testo su WiData. In un progetto reale useresti un PDF vero.

In [ ]:
# Creiamo un documento di testo su WiData
documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

# Salva su file
with open("manuale_widata.txt", "w", encoding="utf-8") as f:
    f.write(documento_widata)

print(f"✅ Documento creato: {len(documento_widata)} caratteri")

✅ Documento creato: 1846 caratteri


---
## 2. Chunking e Indicizzazione

In [ ]:
def chunka_testo(testo, chunk_size=400, overlap=50):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk) #lo aggiunge alla lista dei risultati
        start += chunk_size - overlap #il cursore si sposta in avanti di 400-50 caratteri
    return chunks

chunks = chunka_testo(documento_widata)
print(f"📊 Numero di chunk: {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} char) ---")
    print(chunk+"...")
    print()

📊 Numero di chunk: 6

--- Chunk 1 (400 char) ---

WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2...

--- Chunk 2 (400 char) ---
. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing int...

--- Chunk 3 (400 char) ---
km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Etherne

`start = 0`: La funzione l'inizio del testo al carattere 0.

`while start < len(testo):`: Avvia un ciclo che continua a scorrere il documento finché il cursore non raggiunge la fine del testo.

`end = start + chunk_size`: Calcola dove deve finire il frammento corrente. Di default, conta 400 caratteri in avanti rispetto alla posizione di start.

`chunk = testo[start:end]`: Estrae fisicamente la porzione di testo compresa tra start e end



ChromaDB prende automaticamente i testi leggibili e li traduce in embedding (liste di numeri che rappresentano il significato matematico di quelle parole).
chromadb.Client() crea il database vettoriale temporaneo.

Senza ChromaDB, per fare una domanda a Claude su un intero manuale aziendale dovresti incollare l'intero manuale dentro ogni singola richiesta (spendendo tantissimi soldi in token e rallentando il sistema).

In [ ]:
import chromadb

# Crea il client ChromaDB in memoria
chroma_client = chromadb.Client()

# Crea o recupera la collection
collection = chroma_client.get_or_create_collection(
    name="widata_docs",
    metadata={"hnsw:space": "cosine"}  # usa similarità coseno
)

# Indicizza i chunk
#inserendo i dati dentro la collection tramite il metodo .add()
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ Indicizzati {collection.count()} chunk in ChromaDB")
print("💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!")

✅ Indicizzati 6 chunk in ChromaDB
💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!


`get_or_create_collection`: È un metodo sicuro. Se la collezione "widata_docs" esiste già, la recupera; se non esiste, la crea da zero.

`metadata={"hnsw:space": "cosine"}`: Qui stai impostando la formula matematica che il database userà per capire quanto due testi si somigliano. La similarità coseno misura l'angolo tra due vettori: più l'angolo è stretto (vicino a 0), più i due testi sono semanticamente vicini come significato.

`documents=chunks`: Passi la lista di stringhe creata precedentemente con la funzione di chunking.

`ids=[...]`: Ogni frammento di testo inserito in un database deve avere un codice identificativo unico.

---
## 3. Ricerca Semantica

In [ ]:
#Questa funzione prende la domanda dell'utente e interroga la collezione attiva di ChromaDB.
def cerca(domanda, n_risultati=3):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

# Test ricerca semantica
domande_test = [
    "Quali sensori supportate per ambienti esterni?",
    "Come posso integrare i dati con il mio sistema ERP?",
    "Qual è il costo del piano professionale?",
]

for domanda in domande_test:
    print(f"\n❓ {domanda}")
    chunks_trovati = cerca(domanda, n_risultati=2)
    for i, chunk in enumerate(chunks_trovati):
        print(f"  📄 Chunk {i+1}: {chunk[:120]}...")


❓ Quali sensori supportate per ambienti esterni?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Come posso integrare i dati con il mio sistema ERP?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Qual è il costo del piano professionale?
  📄 Chunk 1: : Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...
  📄 Chunk 2: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...


`query_texts=[domanda]`: ChromaDB prende la stringa della domanda, la trasforma in un vettore numerico e confronta questo vettore con tutti quelli salvati nel database.

`n_results=n_risultati`: Indica al database quanti frammenti restituisce, ordinati dal più rilevante al meno rilevante.

`return risultati["documents"][0]`: ChromaDB restituisce un dizionario complesso contenente ID, distanze matematiche e testi. Scrivendo ["documents"][0], la funzione estrae e restituisce solo la lista pulita dei testi dei chunk trovati per la prima domanda inviata.

---
## 4. RAG Completo — Domanda + Contesto + Risposta

Motore di ricerca semantica (ChromaDB) + Intelligenza Artificiale generativa (Claude)

In [ ]:
SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
Non inventare mai informazioni. Sii conciso e preciso.
"""

def chat_rag(domanda, n_chunks=3):
    """Chatbot con RAG: recupera contesto e genera risposta."""
    # 1. Recupera i chunk rilevanti
    chunks_rilevanti = cerca(domanda, n_risultati=n_chunks)
    contesto = "\n\n---\n\n".join(chunks_rilevanti)

    # 2. Costruisci il prompt aumentato
    prompt = f"""Documenti di riferimento:

      {contesto}

      ---

      Domanda dell'utente: {domanda}"""

    # 3. Genera la risposta
    risposta = chiedi_claude(prompt, system=SYSTEM_WIDATA)
    return risposta, chunks_rilevanti

# Test completo
domanda = "Il sensore XS200 funziona in ambienti molto freddi?"
risposta, chunks = chat_rag(domanda)

print(f"❓ {domanda}")
print(f"\n🤖 {risposta}")
print(f"\n📄 Basato su {len(chunks)} chunk")

❓ Il sensore XS200 funziona in ambienti molto freddi?

🤖 # Risposta

Sì, il sensore XS200 funziona in ambienti freddi. 

Secondo le specifiche tecniche, il sensore è in grado di misurare temperature da **-20°C a +60°C**, quindi è idoneo per ambienti molto freddi fino a -20°C.

Inoltre, grazie alla classificazione **IP67**, il sensore è impermeabile e resistente alla polvere, caratteristiche che lo rendono robusto anche in condizioni ambientali difficili.

📄 Basato su 3 chunk


In [ ]:
# Test con domanda fuori dai documenti
domanda_off = "Quali sono i migliori smartphone del 2025?"
risposta_off, _ = chat_rag(domanda_off)
print(f"❓ {domanda_off}")
print(f"\n🤖 {risposta_off}")
print("\n💡 Il sistema dovrebbe rifiutarsi di rispondere!")

❓ Quali sono i migliori smartphone del 2025?

🤖 Non ho questa informazione nei miei documenti.

I documenti che ho a disposizione riguardano i prodotti e i servizi di WiData Srl (sensori IoT, gateway, piattaforma cloud Xplore, piani di abbonamento e supporto tecnico), non trattano di smartphone.

Se hai domande sui nostri prodotti IoT e smart cities, sarò felice di aiutarti! 😊

💡 Il sistema dovrebbe rifiutarsi di rispondere!


---
## ⭐ Esercizi

In [ ]:
NOME_STUDENTE = ""  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

⚠️ Scrivi il tuo nome!


### Esercizio 1 — Indicizza un documento tuo ★☆☆
Crea un documento di testo su un argomento a tua scelta (può essere anche una dispensa del corso, una ricetta, un regolamento). Indicizzalo in ChromaDB e fai 3 domande. I chunk recuperati sono rilevanti?

In [ ]:
# ESERCIZIO 1
import chromadb

chroma_client = chromadb.Client()


mio_documento = """
L'Intelligenza Artificiale (AI) è una branca dell'informatica che crea sistemi capaci di simulare il cervello umano.
L'obiettivo non è eseguire comandi fissi, ma permettere alle macchine di percepire il mondo esterno,
elaborare le informazioni e agire da sole di conseguenza.

Il cuore dell'AI moderna sono il Machine Learning e il Deep Learning.
Invece di seguire regole rigide decise da un programmatore, questi sistemi analizzano montagne di dati
per trovare pattern nascosti, apprendere dai propri errori e migliorare nel tempo in modo del tutto autonomo.

Oggi l'AI si usa in tantissimi campi della vita quotidiana. Ad esempio, permette alle auto a guida autonoma
 di riconoscere i semafori, aiuta i medici a scoprire malattie dalle radiografie
 e fa funzionare i filtri anti-spam che bloccano le email pubblicitarie fastidiose nella tua posta.
"""

print(f"✅ Documento creato: {len(mio_documento)} caratteri")

# Crea una nuova collection (o recupera)
mia_collection = chroma_client.get_or_create_collection(name="mio_doc_v2", metadata={"hnsw:space": "cosine"})

def chunka_testo(testo, chunk_size=100, overlap=25):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def cerca(domanda, n_risultati=3):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = mia_collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

# Chunka e indicizza
chunks = chunka_testo(mio_documento)

# Indicizza i chunk
mia_collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ Indicizzati {mia_collection.count()} chunk in ChromaDB")


# Fai 3 domande e stampa i chunk recuperati
domande_test = [
    "Qual è il vero obiettivo dell'Intelligenza Artificiale?",  # Pescherà solo il CHUNK 1
    "In che modo i sistemi di AI riescono a migliorare da soli?", # Pescherà solo il CHUNK 2
    "Quali sono alcuni esempi pratici di utilizzo dell'AI?",     # Pescherà solo il CHUNK 3
    "antispam è un applicazione di AI?"
]

for domanda in domande_test:
    print(f"\n❓ {domanda}")
    chunks_trovati = cerca(domanda, n_risultati=2)
    for i, chunk in enumerate(chunks_trovati):
        print(f"  📄 Chunk {i+1}: {chunk}...")
        print("-------------------------------------")
    print("\n")

✅ Documento creato: 856 caratteri
✅ Indicizzati 12 chunk in ChromaDB

❓ Qual è il vero obiettivo dell'Intelligenza Artificiale?
  📄 Chunk 1: 
L'Intelligenza Artificiale (AI) è una branca dell'informatica che crea sistemi capaci di simulare il cervello umano.
L'obiettivo non è eseguire comandi fissi, ma permettere alle macchine di percepire il mondo esterno, 
elaborare le informazioni e agire da sole di conseguenza.

Il cuore dell'AI mode...
-------------------------------------
  📄 Chunk 2: ire da sole di conseguenza.

Il cuore dell'AI moderna sono il Machine Learning e il Deep Learning. 
Invece di seguire regole rigide decise da un programmatore, questi sistemi analizzano montagne di dati 
per trovare pattern nascosti, apprendere dai propri errori e migliorare nel tempo in modo del tu...
-------------------------------------



❓ In che modo i sistemi di AI riescono a migliorare da soli?
  📄 Chunk 1: ti 
per trovare pattern nascosti, apprendere dai propri errori e migliorare nel tempo

### Esercizio 2 — Sperimenta con il chunking ★★☆
Prova a indicizzare lo stesso documento con chunk_size=200, 400 e 800. Per la stessa domanda, i chunk recuperati sono diversi? Quale dimensione dà risultati migliori?

In [ ]:
#DA RICORDARE

import chromadb

# Crea il client ChromaDB in memoria
chroma_client = chromadb.Client()

def chunka_testo(testo, chunk_size=400, overlap=60):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def cerca(domanda, n_risultati=1):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

mio_documento = """
L'Intelligenza Artificiale (AI) è una branca dell'informatica che crea sistemi capaci di simulare il cervello umano.
L'obiettivo non è eseguire comandi fissi, ma permettere alle macchine di percepire il mondo esterno,
elaborare le informazioni e agire da sole di conseguenza.

Il cuore dell'AI moderna sono il Machine Learning e il Deep Learning.
Invece di seguire regole rigide decise da un programmatore, questi sistemi analizzano montagne di dati
per trovare pattern nascosti, apprendere dai propri errori e migliorare nel tempo in modo del tutto autonomo.

Oggi l'AI si usa in tantissimi campi della vita quotidiana. Ad esempio, permette alle auto a guida autonoma
 di riconoscere i semafori, aiuta i medici a scoprire malattie dalle radiografie
 e fa funzionare i filtri anti-spam che bloccano le email pubblicitarie fastidiose nella tua posta.
"""

In [ ]:
# ESERCIZIO 2
domanda_test = "Quali sono alcuni esempi pratici di utilizzo dell'AI?"

for chunk_size in [200, 400, 800]:
    print(f"\n{'='*50}")
    print(f"chunk_size = {chunk_size}")
    print('='*50)

    # TODO: crea collection, chunka con dimensione diversa, indicizza, cerca
    nome_collection = f"mio_doc_{chunk_size}"
    collection = chroma_client.get_or_create_collection(
        name=nome_collection,
        metadata={"hnsw:space": "cosine"}
        )

    overlap_dinamico=int(chunk_size/4)
    chunks = chunka_testo(mio_documento, chunk_size=chunk_size, overlap=overlap_dinamico)
    # Indicizza i chunk
    collection.add(
        documents=chunks,
        ids=[f"chunk_{i}" for i in range(len(chunks))]
        )
    print(f"✅ Indicizzati {collection.count()} chunk nella collection '{nome_collection}' in ChromaDB")


    risultati = cerca(domanda_test)

    # stampa i risultati
    print(f"\n🎯 Risultato della ricerca per chunk_size {chunk_size}:")
    testo = risultati[0].replace("\n", " ").strip()

    print(f"\"{testo}...\"")
    pass

# Commento: quale chunk_size ha dato i risultati migliori?
# Risposta: ...



chunk_size = 200
✅ Indicizzati 6 chunk nella collection 'mio_doc_200' in ChromaDB

🎯 Risultato della ricerca per chunk_size 200:
"L'Intelligenza Artificiale (AI) è una branca dell'informatica che crea sistemi capaci di simulare il cervello umano. L'obiettivo non è eseguire comandi fissi, ma permettere alle macchine di percepire..."

chunk_size = 400
✅ Indicizzati 3 chunk nella collection 'mio_doc_400' in ChromaDB

🎯 Risultato della ricerca per chunk_size 400:
"L'Intelligenza Artificiale (AI) è una branca dell'informatica che crea sistemi capaci di simulare il cervello umano. L'obiettivo non è eseguire comandi fissi, ma permettere alle macchine di percepire il mondo esterno,  elaborare le informazioni e agire da sole di conseguenza.  Il cuore dell'AI moderna sono il Machine Learning e il Deep Learning.  Invece di seguire regole rigide decise da un progr..."

chunk_size = 800
✅ Indicizzati 2 chunk nella collection 'mio_doc_800' in ChromaDB

🎯 Risultato della ricerca per chunk_size 800:


Il chunk_size di 400 ha dato il risultato migliore.
#
A 200 il testo viene frammentato troppo rischiando di tagliare il contesto a metà frase. Ha mancato il bersaglio! Questo perché, essendo i pezzi troppo piccoli, il database si è fatto ingannare da altre parole simili e ha estratto il blocco teorico sbagliato, lasciando fuori gli esempi.

A 800 il chunk è troppo grande e restituisce l'intero documento, vanificando la precisione del recupero. Include un sacco di testo superfluo, troppo grande e impreciso per questa lunghezza di documento.

La dimensione a 400 permette di isolare il paragrafo degli esempi pratici in modo pulito ed efficiente.

### Esercizio 3 — RAG + storia conversazione ★★☆
Integra RAG nella funzione `chat()` della Lezione 3 (quella con la history). Ogni risposta deve usare sia il contesto RAG che la storia della conversazione.

In [ ]:
def chat(messaggio, system=None):
    """Invia un messaggio mantenendo la history."""
    # 1. Aggiungi il messaggio dell'utente
    history.append({"role": "user", "content": messaggio})

    # 2. Invia TUTTA la history al modello
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 500,
        "messages": history
    }
    if system:
        params["system"] = system

    risposta = client.messages.create(**params)
    testo = risposta.content[0].text

    # 3. Aggiungi la risposta alla history
    history.append({"role": "assistant", "content": testo})

    return testo


def cerca(domanda, n_risultati=1):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
Non inventare mai informazioni. Sii conciso e preciso.
"""


In [ ]:
# ESERCIZIO 3
history = []

def chat_rag_con_storia(domanda):
    """Chatbot con RAG + conversazione multi-turno."""
    # TODO:
    # 1. Recupera chunk rilevanti con cerca()
    chunks_rilevanti = cerca(domanda, n_risultati=2)
    contesto = "\n---\n".join(chunks_rilevanti)

    # 2. Costruisci il messaggio con contesto + domanda
    prompt_con_contesto = f"""Documenti di riferimento:

    {contesto}

    ---

    Domanda dell'utente: {domanda}"""

    # 3. Aggiungi alla history
    # 4. Chiama l'API con tutta la history
    # 5. Aggiungi la risposta alla history
    risposta = chat(prompt_con_contesto, system=SYSTEM_WIDATA)

    # 6. Restituisci la risposta
    return risposta

# Creiamo un documento di testo su WiData
documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

nome_collection_widata = "widata_docs_es3"
collection = chroma_client.get_or_create_collection(
    name=nome_collection_widata,
    metadata={"hnsw:space": "cosine"}
)

chunks = chunka_testo(documento_widata)

collection.add(
    documents=chunks,
    ids=[f"chunk_widata_{i}" for i in range(len(chunks))]
)
print(f"✅ Nuovi chunk caricati con successo in ChromaDB!")
print(f"La collezione '{nome_collection_widata}' contiene ora {collection.count()} elementi.\n")



# Test: domande collegate che richiedono sia RAG che memoria
# chat_rag_con_storia("Parlami del sensore XS200")
# chat_rag_con_storia("Qual è la sua autonomia?")
# chat_rag_con_storia("Costa molto?")


# Domanda 1: Richiede solo il RAG (estrazione info dal testo)
print("❓ Utente: Parlami del sensore XS200")
risposta1 = chat_rag_con_storia("Parlami del sensore XS200")
print(f"🤖 Claude: {risposta1}\n")

# Domanda 2: Richiede il RAG + la Memoria (deve sapere cos'è "sua" riferito a XS200)
print("❓ Utente: Qual è la sua autonomia?")
risposta2 = chat_rag_con_storia("Qual è la sua autonomia?")
print(f"🤖 Claude: {risposta2}\n")

# Domanda 3: Richiede Memoria + RAG (deve capire che parliamo ancora del sensore e cercare i prezzi)
print("❓ Utente: Costa molto?")
risposta3 = chat_rag_con_storia("Costa molto?")
print(f"🤖 Claude: {risposta3}\n")

✅ Nuovi chunk caricati con successo in ChromaDB!
La collezione 'widata_docs_es3' contiene ora 6 elementi.

❓ Utente: Parlami del sensore XS200
🤖 Claude: # Sensore XS200 - Monitoraggio Ambientale

Il sensore **XS200** è progettato per il monitoraggio ambientale in ambienti industriali e urbani.

## Caratteristiche tecniche:

**Parametri misurati:**
- Temperatura: da -20°C a +60°C
- Umidità relativa: 0-100%
- Pressione atmosferica
- Qualità dell'aria: CO2 e PM2.5

**Resistenza e durabilità:**
- Classificazione **IP67**: impermeabile e resistente alla polvere

**Alimentazione:**
- Batteria Li-Ion 3.7V
- Autonomia: 2 anni (il documento è parzialmente tagliato)

Il sensore è ideale per applicazioni di monitoraggio ambientale che richiedono robustezza e affidabilità in condizioni esterne difficili.

❓ Utente: Qual è la sua autonomia?
🤖 Claude: Non ho questa informazione nei miei documenti.

I documenti forniti descrivono due prodotti diversi:
1. Il **sensore XS200** - di cui conosco l'autono

### Esercizio 4 — Chatbot RAG WiData completo ★★★ (Deliverable!)

Costruisci il chatbot completo con:
- RAG sul documento WiData
- Conversation history (sliding window)
- Streaming
- System prompt WiData con istruzione anti-hallucination
- Loop interattivo con `input()`
- Stampa i chunk usati per ogni risposta (per debug)

In [ ]:
# ESERCIZIO 4 — Chatbot RAG completo (DELIVERABLE)
import chromadb, json, os
from google.colab import userdata
import anthropic

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
chroma_client = chromadb.Client()

SYSTEM = """
Sei l'assistente virtuale ufficiale di WiData Srl, azienda IoT e smart cities con sede a Sassari.
Il tuo compito è aiutare gli utenti rispondendo esclusivamente sulla base dei "Documenti di riferimento" forniti di volta in volta.

Regole tassative anti-allucinazione:
1. Rispondi SOLO se l'informazione è esplicitamente presente nei documenti di riferimento forniti.
2. Se la risposta non si trova nei documenti, dì testualmente ed esclusivamente: 'Non ho questa informazione nei miei documenti.'
3. Non tentare di inventare, estrapolare o dedurre informazioni non scritte (es. prezzi non specificati o specifiche tecniche mancanti).
4. Sii conciso, professionale e preciso.
"""

MAX_MESSAGGI = 10

def setup_rag(testo):
    """Indicizza il documento e restituisce la collection."""
    nome_coll = "widata_collection"
    collection = chroma_client.get_or_create_collection(
        name=nome_coll,
        metadata={"hnsw:space": "cosine"}
    )
    chunks = chunka_testo(testo)
    collection.add(
        documents=chunks,
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )
    print(f"📊 RAG Configurato: {collection.count()} chunk logici indicizzati con successo.")
    return collection

def cerca(domanda, collection, n=3):
    """Ricerca semantica nella collection."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n
    )
    return risultati["documents"][0]


def chat_completo(domanda, history, collection):
    """Chatbot con RAG + storia + streaming."""
    chunks_rilevanti = cerca(domanda, collection, n=2)
    contesto = "\n---\n".join(chunks_rilevanti)

    prompt_con_contesto = f"""Documenti di riferimento:
      {contesto}

      ---
      Domanda dell'utente: {domanda}"""

    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]


    messaggi_da_inviare = list(history)
    messaggi_da_inviare.append({"role": "user", "content": prompt_con_contesto})

    print("\n🤖 Assistente: ", end="", flush=True)


    testo_risposta = ""
    with client.messages.stream(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        system=SYSTEM,
        messages=messaggi_da_inviare
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
            testo_risposta += text
    print("\n") # Va a capo a fine streaming

    history.append({"role": "user", "content": domanda})
    history.append({"role": "assistant", "content": testo_risposta})


documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

def main():
    collection = setup_rag(documento_widata)
    history = []
    print("🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        chat_completo(utente, history, collection)

main()

📊 RAG Configurato: 6 chunk logici indicizzati con successo.
🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.

Tu: dimmi le specifiche del sensore XS200

🤖 Assistente: # Specifiche del Sensore XS200

Il sensore **XS200** è progettato per il monitoraggio ambientale in ambienti industriali e urbani.

**Parametri misurati:**
- Temperatura: da -20°C a +60°C
- Umidità relativa: 0-100%
- Pressione atmosferica
- Qualità dell'aria (CO2, PM2.5)

**Caratteristiche fisiche:**
- Classificazione IP67: impermeabile e resistente alla polvere

**Alimentazione:**
- Batteria Li-Ion 3.7V

**Nota:** Il documento di riferimento non specifica completamente l'autonomia della batteria (il testo risulta troncato).

Tu: come posso avere assistenza?

🤖 Assistente: # Come Avere Assistenza

WiData offre supporto tecnico con i seguenti contatti:

**Email:** support@widata.cloud
**Telefono:** +39 079 123456

**Orari di disponibilità:** Lunedì-venerdì, 9:00-18:00

**Per informazioni commerciali:** sales@widata.

---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione4_TUONOME.ipynb`
4. Carica su GitHub in `lezione4/`

```bash
git add lezione4/
git commit -m "Lezione 4 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 04/06)
Leggi **Huyen Cap. 6 — sezione Agents**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*